In [1]:
import os
import shutil
import random
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict
import torch
from ultralytics import YOLO

print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

DATASET_ROOT = r"D:\data"   

RAW_IMAGES_PATH = os.path.join(DATASET_ROOT, "images")
RAW_LABELS_PATH = os.path.join(DATASET_ROOT, "labels")
YOLO_DS_PATH    = os.path.join(DATASET_ROOT, "yolov8_dataset")

print(f"Images folder exists: {os.path.exists(RAW_IMAGES_PATH)}")
print(f"Labels folder exists: {os.path.exists(RAW_LABELS_PATH)}")

CUDA Available : True
GPU            : NVIDIA GeForce RTX 2050
Images folder exists: True
Labels folder exists: True


In [2]:

RAW_TO_ENGLISH = {
'1_chongkong' : 'punching_hole',
'2_hanfeng' : 'welding_line',
'3_yueyawan' : 'crescent_gap',
'4_shuiban' : 'water_spot',
'5_youban' : 'oil_spot',
'6_siban' : 'silk_spot',
'7_yiwu' : 'inclusion',
'8_yahen' : 'dent',
'9_zhehen' : 'rolled_pit',
'10_yaozhed' : 'waist_folding',
}

CLASS_NAMES = sorted(set(RAW_TO_ENGLISH.values()))
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

print("Starting to build YOLO dataset...")

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(YOLO_DS_PATH, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DS_PATH, 'labels', split), exist_ok=True)

img_map = {}
for root, _, files in os.walk(RAW_IMAGES_PATH):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_map[f] = os.path.join(root, f)

print(f"Found {len(img_map)} images")

img_labels = defaultdict(list)

xml_files = [os.path.join(root, f) for root, _, files in os.walk(RAW_LABELS_PATH)
             for f in files if f.endswith('.xml')]

print(f"Parsing {len(xml_files)} XML files...")

for xml_path in xml_files:
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        filename = root.find('filename').text if root.find('filename') is not None else None
       
        size = root.find('size')
        iw = int(size.find('width').text) if size is not None else None
        ih = int(size.find('height').text) if size is not None else None

        for obj in root.findall('object'):
            name_raw = obj.find('name').text.strip()
            if name_raw in RAW_TO_ENGLISH:
                cls_id = CLASS_TO_ID[RAW_TO_ENGLISH[name_raw]]
                bb = obj.find('bndbox')
                xmin = int(float(bb.find('xmin').text))
                ymin = int(float(bb.find('ymin').text))
                xmax = int(float(bb.find('xmax').text))
                ymax = int(float(bb.find('ymax').text))

                if iw and ih:
                    xc = ((xmin + xmax) / 2) / iw
                    yc = ((ymin + ymax) / 2) / ih
                    w = (xmax - xmin) / iw
                    h = (ymax - ymin) / ih
                    img_labels[filename].append((cls_id, xc, yc, w, h))
    except:
        continue


valid_imgs = list(img_labels.keys())
random.shuffle(valid_imgs)

n = len(valid_imgs)
train_imgs = valid_imgs[:int(n*0.7)]
val_imgs   = valid_imgs[int(n*0.7):int(n*0.9)]
test_imgs  = valid_imgs[int(n*0.9):]

print(f"Split → Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")


for imgs, split in zip([train_imgs, val_imgs, test_imgs], ['train', 'val', 'test']):
    for img_fn in imgs:
        src_img = img_map.get(img_fn)
        if src_img and os.path.exists(src_img):
            dst_img = os.path.join(YOLO_DS_PATH, 'images', split, img_fn)
            shutil.copy2(src_img, dst_img)

        stem = Path(img_fn).stem
        lbl_path = os.path.join(YOLO_DS_PATH, 'labels', split, stem + '.txt')
        with open(lbl_path, 'w') as f:
            for cls_id, xc, yc, w, h in img_labels[img_fn]:
                f.write(f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

print("YOLO dataset built successfully!")
print(f"Location: {YOLO_DS_PATH}")

Starting to build YOLO dataset...
Found 3802 images
Parsing 3802 XML files...
Split → Train: 2661 | Val: 760 | Test: 381
✅ YOLO dataset built successfully!
Location: D:\data\yolov8_dataset


In [3]:
import yaml
import os

data_yaml = {
    'path': YOLO_DS_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES
}

yaml_path = os.path.join(YOLO_DS_PATH, 'data.yaml')

with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print("data.yaml created successfully!")
print("\nContent of data.yaml:")
print(open(yaml_path).read())

✅ data.yaml created successfully!

Content of data.yaml:
path: D:\data\yolov8_dataset
train: images/train
val: images/val
test: images/test
nc: 10
names:
- crescent_gap
- dent
- inclusion
- oil_spot
- punching_hole
- rolled_pit
- silk_spot
- waist_folding
- water_spot
- welding_line



In [4]:

from ultralytics import YOLO
import torch
import os

print("=== TRAINING SETUP ===")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nTraining on: {len(os.listdir(os.path.join(YOLO_DS_PATH, 'images', 'train')))} images")
print(f"Validation on: {len(os.listdir(os.path.join(YOLO_DS_PATH, 'images', 'val')))} images")

# Load model
model = YOLO("yolov8s.pt")

print("\nStarting Training...\n")

results = model.train(
    data=yaml_path,
    epochs=100,              
    imgsz=640,
    batch=8,              
    device=0,              
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    workers=6,
   
    hsv_h=0.0,      
    hsv_s=0.0,      
    hsv_v=0.5,     
    degrees=8.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
   
    project=os.path.join(DATASET_ROOT, "runs"),
    name="yolov8n_steel_defect",
    exist_ok=True,
    verbose=True
)

print("\nTRAINING FINISHED!")
print(f"Best model saved at: {results.save_dir}/weights/best.pt")

=== TRAINING SETUP ===
GPU Available: True
GPU: NVIDIA GeForce RTX 2050

Training on: 3467 images
Validation on: 1386 images

🚀 Starting Training...

Ultralytics 8.4.37  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\data\yolov8_dataset\data.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov